This colab notebook performs inference with a model trained on GOLEMcoref described in our ACL 2026 paper.

### Clone Github repo

In [1]:
%%capture
! git clone https://github.com/andreasvc/fast-coref.git

### Install Relevant Libraries

All required libraries are installed by default on Google Colab. Below is a list of versions that have been tested to work, for documentation purposes.

In [12]:
versions  = """
%%capture
! pip install torch==2.10.0
! pip install transformers==5.2.0
! pip install scipy==1.14.1
! pip install omegaconf==2.3.0
! pip install 'huggingface-hub>=1.2.0'
! pip install spacy==3.8.7
! python -m spacy download en_core_web_sm
"""

### Download Pretrained Model

Download the crosslingual fast-coref model trained on GOLEMcoref.

In [3]:
%%bash
# Download the model, but only if the directory does not exist
if [ ! -d "golem_joint" ]; then
  curl https://urd2.let.rug.nl/~andreas/golem_joint.tar | tar -x
fi

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1337M  100 1337M    0     0  17.5M      0  0:01:16  0:01:16 --:--:-- 20.8M


In [4]:
import sys
sys.path.append('fast-coref/src')
from inference.model_inference import Inference

# Load the model
inference_model = Inference("golem_joint/", encoder_name="golem_joint/doc_encoder/")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [5]:
ls

fast-coref/  golem_joint/  sample_data/


### Sample Doc

In [6]:
doc = """
    Bilbo celebrates his eleventy-first birthday and leaves the Shire suddenly, passing the Ring to Frodo Baggins, his cousin and heir.
    Neither hobbit is aware of the Ring's origin, but the wizard Gandalf suspects it is a Ring of Power.
    Seventeen years later, Gandalf tells Frodo that he has confirmed that the Ring is the one lost by the Dark Lord Sauron long ago and counsels him to take it away from the Shire.
    """

In [7]:
output = inference_model.perform_coreference(doc)

#### Remove Singletons


In [8]:
for cluster in output["clusters"]:
  if len(cluster) > 1:
    print(cluster)

[((6, 7), 'Bilbo'), ((9, 9), 'his'), ((30, 30), 'his')]
[((25, 28), 'Frodo Baggins'), ((76, 77), 'Frodo'), ((99, 99), 'him')]
[((55, 55), 'Gandalf'), ((74, 74), 'Gandalf'), ((79, 79), 'he')]


####All Clusters

In [9]:
for cluster in output["clusters"]:
  print(cluster)

[((6, 7), 'Bilbo'), ((9, 9), 'his'), ((30, 30), 'his')]
[((25, 28), 'Frodo Baggins'), ((76, 77), 'Frodo'), ((99, 99), 'him')]
[((30, 33), 'his cousin and heir')]
[((55, 55), 'Gandalf'), ((74, 74), 'Gandalf'), ((79, 79), 'he')]
[((90, 94), 'the Dark Lord Sauron')]
[((93, 94), 'Sauron')]


In [10]:
# This is a multilingual model, which also speaks Dutch.
doc = """Izzy wordt regelmatig geconfronteerd met het feit dat hij niet de
langste persoon is in zijn nogal uitgebreide huishouden, maar als hij de
kandidatenlijst openvouwt dat net door de gemeente is bezorgd, zucht hij diep.
Het stembiljet laat hem altijd nog kleiner voelen dan hij al is, en dit jaar
is erger dan de afgelopen paar keren.
"""
output = inference_model.perform_coreference(doc)
for cluster in output["clusters"]:
  if len(cluster) > 1:
    print(cluster)

[((0, 0), 'Izzy'), ((12, 12), 'hij'), ((22, 22), 'zijn'), ((32, 32), 'hij'), ((54, 54), 'hij'), ((64, 64), 'hem'), ((70, 70), 'hij')]
